In [70]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("engine")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "/home/jovyan/notebooks/lakehouse")
    .getOrCreate()
)

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.db_lakehouse")



DataFrame[]

# 1.-Extraccion

In [71]:

jdbc_url = "jdbc:postgresql://postgres:5432/db_acatlan"
props = {
    "user": "acatlan",
    "password": "8g4izJPX",
    "driver": "org.postgresql.Driver",
}

In [72]:
df_order_items = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.order_items")
    .option("user", props["user"])
    .option("password", props["password"])
    .option("driver", props["driver"])
    .load()
)

In [73]:
df_orders = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.orders")
    .option("user", props["user"])
    .option("password", props["password"])
    .option("driver", props["driver"])
    .load()
)

In [75]:
df_customers = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.customers")
    .option("user", props["user"])
    .option("password", props["password"])
    .option("driver", props["driver"])
    .load()
)

In [83]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: long (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



# 2.-Transformacion

In [78]:
df_staging = df_order_items.join(df_orders,on="order_id",how="inner")

In [82]:
df_staging.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: float (nullable = true)
 |-- freight_value: float (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



In [84]:
df_staging = df_staging.join(df_customers,on="customer_id",how="inner")

In [85]:
df_staging.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: float (nullable = true)
 |-- freight_value: float (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: long (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [87]:
df_staging.show(2,vertical=True)

-RECORD 0---------------------------------------------
 customer_id                   | 00050bf6e01e69d5c... 
 order_id                      | fa906f338cee30a98... 
 order_item_id                 | 1                    
 product_id                    | 79ad39409fa4bcc36... 
 seller_id                     | 1da3aeb70d7989d1e... 
 shipping_limit_date           | 2017-09-21 16:15:13  
 price                         | 69.99                
 freight_value                 | 15.24                
 order_status                  | delivered            
 order_purchase_timestamp      | 2017-09-17 16:04:44  
 order_approved_at             | 2017-09-17 16:15:13  
 order_delivered_carrier_date  | 2017-09-18 21:02:46  
 order_delivered_customer_date | 2017-10-02 21:14:31  
 order_estimated_delivery_date | 2017-10-13 00:00:00  
 customer_unique_id            | e3cf594a99e810f58... 
 customer_zip_code_prefix      | 98700                
 customer_city                 | ijui                 
 customer_

# 3.-Carga

In [88]:
df_staging.writeTo("local.db_lakehouse.staging_data").createOrReplace()

# 4.-Acid

In [95]:
spark.sql("SELECT * FROM local.db_lakehouse.staging_data LIMIT 4").show(vertical=True,truncate=False)

(0 rows)



In [92]:
spark.sql("""
UPDATE local.db_lakehouse.staging_data
SET price = 150000.0,
    freight_value = 20000.0
WHERE customer_id = '00050bf6e01e69d5c0fd612f1bcfb69c'
""")

DataFrame[]

In [94]:
spark.sql("""
TRUNCATE TABLE local.db_lakehouse.staging_data
""")

DataFrame[]